In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error

### 회귀 모델용 성능 평가 함수 사용

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression

housing = fetch_california_housing()
X = housing.data
y = housing.target

In [3]:
X

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8))

In [5]:
# 캘리포니아 지역에 있는 집값
# 단위 : 100,000달러(예 2.0 = $200,000)
y

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

In [6]:
# 학습 데이터와 테스트 데이터로 분할한다.(8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# 학습
model = LinearRegression()
model.fit(X_train, y_train)
# 예측
y_pred = model.predict(X_test)

In [9]:
# 성능 평가 지표
# MAE
mae = mean_absolute_error(y_test,y_pred)
# MSE
mse = mean_squared_error(y_test,y_pred)
# RMSE
rmse = root_mean_squared_error(y_test,y_pred)

In [12]:
print(f'MAE : {mae:.4f}')
print(f'MSE : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')


MAE : 0.5332
MSE : 0.5559
RMSE : 0.7456


### 이상치가 있을 때

In [13]:
housing = fetch_california_housing()
y_true = housing.target[:100]
y_pred = y_true.copy()

In [17]:
# 상황 A : 아주 우수한 모델 (모든 데이터에 평균 0.1의  작은 오차만 있는 경우)
y_pred_normal = y_true + np.random.normal(0,0.1, size=y_true.shape)

mae1 = mean_absolute_error(y_true, y_pred_normal)
mse1 = mean_squared_error(y_true, y_pred_normal)
rmse1 = root_mean_squared_error(y_true, y_pred_normal)

In [18]:
# 상황 B : 이상치 발생(99개는 오차가 똑같지만 딱 1개를 100배의 오차로 틀리게~)
y_pred_outlier = y_pred_normal.copy()
y_pred_outlier[0] = y_pred_outlier[0] + 20.0

mae2 = mean_absolute_error(y_true, y_pred_outlier)
mse2 = mean_squared_error(y_true, y_pred_outlier)
rmse2 = root_mean_squared_error(y_true, y_pred_outlier)

In [19]:
print("상황 A : 모든 오차가 비슷함")
print(f'MAE: {mae1:.4f}')
print(f'MSE: {mse1:.4f}')
print(f'RMSE: {rmse1:.4f}')
print("상황 B : 딱 하나만 오차가 매우크고 나머지는 비슷함")
print(f'MAE: {mae2:.4f}')
print(f'MSE: {mse2:.4f}')
print(f'RMSE: {rmse2:.4f}')

상황 A : 모든 오차가 비슷함
MAE: 0.0871
MSE: 0.0118
RMSE: 0.1087
상황 B : 딱 하나만 오차가 매우크고 나머지는 비슷함
MAE: 0.2844
MSE: 3.9578
RMSE: 1.9894


- MAE : 최종 평가용으로 사용하는 오차. 실제 오차의 평균
- MSE : 학습 단계에서 사용하는 오차.
- RMSE : MSE에 루트를 씌워서 단위를 MAE와 동일하게 맞춘것.
- 만약 MAE와 RMSE가 매우 비슷하다면 : 전체 데이터에 대한 오차가 다 비슷한 상황
- 만약 RMSE가 MAE보다 훨씬 크다면 : 전체 데이터 중 다른 데이터 비해 오차가 매우 큰 데이터가 있다는 것으로 해석
- MAE와 RMSE를 같이보는 걸 선호